# Spatial Block Split — Harbour Basin and Northern Channels

Runs the spatially disjoint block-split evaluation in both directions: harbour basin train / northern channels test, and northern channels train / harbour basin test.

**Project:** Development and Evaluation of a Deep Learning Model for Flood Detection and Drought Prediction Using Satellite Remote Sensing Data in South Africa
**Study Area:** KwaZulu-Natal, South Africa
**Flood Event:** April 2022 KwaZulu-Natal Floods
**Reference Product:** UNOSAT FL20220418ZAF
**Student:** Athindothe Valencia Marubini
**Student No:** 219160643
**Supervisor:** Prof. IE Davidson
**Co-Supervisor:** Dr O.P Babalola
**Institution:** Cape Peninsula University of Technology (CPUT)

Patch positions are seeded (np.random.seed(42)) for reproducibility. Model training uses full determinism seeding (torch.manual_seed, deterministic cuDNN, num_workers=0). Extraction parameters, the augmentation strategy, and the U-Net architecture and training hyperparameters are identical between directions.

Input: Stacked_4tile/flood_stack_4tile_v1.tif, Mosaic_4tile/flood_label_mosaic.tif
Output: Patches_spatial_split/, Models_spatial_split/, Chapter4_Figures_spatial_split/

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install rasterio torch torchvision scikit-learn --quiet

In [ ]:
import os, gc, zipfile, time, random
import numpy as np
import rasterio
from rasterio.windows import Window
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, cohen_kappa_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cpu':
    print('WARNING: No GPU — training will be slow.')

ROOT       = '/content/drive/MyDrive/KZN_Research_Colab/'
STACK_PATH = ROOT + 'Stacked_4tile/flood_stack_4tile_v1.tif'
LABEL_PATH = ROOT + 'Mosaic_4tile/flood_label_mosaic.tif'

PATCH_DIR  = ROOT + 'Patches_combined_split/'
ZIP_DIR    = ROOT + 'Patches_combined_split_zips/'
MODEL_DIR  = ROOT + 'Models_combined_split/'
FIG_DIR    = ROOT + 'Chapter4_Figures_combined_split/'
LOCAL_DIR  = '/content/patches_combined_split/'

for d in [PATCH_DIR, ZIP_DIR, MODEL_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

PATCH_SIZE          = 128
STRIDE              = 32
MIN_FLOOD           = 1
RATIO               = 10
MAX_NODATA_FRACTION = 0.3

# Full flood-region extent
FLOOD_ROW_MIN = 9493
FLOOD_ROW_MAX = 12108
FLOOD_COL_MIN = 9061
FLOOD_COL_MAX = 11952

# Northern channels box
NORTH_R0 = 9493
NORTH_R1 = 10635
NORTH_C0 = 11037
NORTH_C1 = 11952

print('Northern channels box: rows', NORTH_R0, '-', NORTH_R1, ' cols', NORTH_C0, '-', NORTH_C1)
print('Full flood region    : rows', FLOOD_ROW_MIN, '-', FLOOD_ROW_MAX, ' cols', FLOOD_COL_MIN, '-', FLOOD_COL_MAX)
print('Harbour basin = full flood region minus the northern channels box.')

Device: cuda
Northern channels box: rows 9493 - 10635  cols 11037 - 11952
Full flood region    : rows 9493 - 12108  cols 9061 - 11952
Harbour basin = full flood region minus the northern channels box.


---
## Step 2: Load Label and Verify Pixel Counts for Both Regions

In [ ]:
print('Loading UNOSAT label...')
with rasterio.open(LABEL_PATH) as src:
    label_full = src.read(1).astype(np.uint8)
    H, W = src.height, src.width

total = int((label_full == 1).sum())
north_flood = int((label_full[NORTH_R0:NORTH_R1, NORTH_C0:NORTH_C1] == 1).sum())

harbour_region = label_full[FLOOD_ROW_MIN:FLOOD_ROW_MAX,
                            FLOOD_COL_MIN:FLOOD_COL_MAX].copy()
harbour_region[NORTH_R0-FLOOD_ROW_MIN:NORTH_R1-FLOOD_ROW_MIN,
               NORTH_C0-FLOOD_COL_MIN:NORTH_C1-FLOOD_COL_MIN] = 0
harbour_flood = int((harbour_region == 1).sum())

print(f'  Total flood pixels           : {total:,}')
print(f'  Northern channels box flood  : {north_flood:,}  ({100*north_flood/total:.1f}%)')
print(f'  Harbour basin flood          : {harbour_flood:,}  ({100*harbour_flood/total:.1f}%)')
print()


Loading UNOSAT label...
  Total flood pixels           : 98,715
  Northern channels box flood  : 12,765  (12.9%)
  Harbour basin flood          : 85,945  (87.1%)



---
## Step 3: Build Nodata Mask (shared by both directions)

In [ ]:
print('Building nodata mask...')
DOWNSAMPLE = 8
with rasterio.open(STACK_PATH) as src:
    h_ds = H // DOWNSAMPLE
    w_ds = W // DOWNSAMPLE
    vv_post_ds = src.read(11, out_shape=(h_ds, w_ds))

assert vv_post_ds.ndim == 2
nodata_mask_ds = (vv_post_ds == 0)

def has_too_much_nodata(row, col):
    r0 = row // DOWNSAMPLE; r1 = (row + PATCH_SIZE) // DOWNSAMPLE
    c0 = col // DOWNSAMPLE; c1 = (col + PATCH_SIZE) // DOWNSAMPLE
    patch_mask = nodata_mask_ds[r0:r1, c0:c1]
    if patch_mask.size == 0: return False
    return patch_mask.mean() > MAX_NODATA_FRACTION

del vv_post_ds
gc.collect()
print('Nodata mask ready.')

Building nodata mask...
Nodata mask ready.


---
## Step 4: Shared Helper Functions (augmentation, position collection, extraction, U-Net)

In [ ]:
def augment(X, y, k):
    if k == 0: return X, y
    if k == 1: return np.rot90(X,1,axes=(1,2)), np.rot90(y,1)
    if k == 2: return np.rot90(X,2,axes=(1,2)), np.rot90(y,2)
    if k == 3: return np.rot90(X,3,axes=(1,2)), np.rot90(y,3)
    if k == 4: return X[:,:,::-1].copy(), y[:,::-1].copy()
    if k == 5: return X[:,::-1,:].copy(), y[::-1,:].copy()
    if k == 6: return np.rot90(X,1,axes=(1,2))[:,:,::-1].copy(), np.rot90(y,1)[:,::-1].copy()
    if k == 7: return np.rot90(X,1,axes=(1,2))[:,::-1,:].copy(), np.rot90(y,1)[::-1,:].copy()
    return X, y

def collect_positions(row_start, row_end, col_start, col_end, exclude_box=None):
    """
    Collect flood and non-flood patch positions.
    exclude_box: (r0,r1,c0,c1) absolute coords to skip.
    """
    flood_pos    = []
    nonflood_pos = []
    for r in range(row_start, row_end - PATCH_SIZE + 1, STRIDE):
        for c in range(col_start, col_end - PATCH_SIZE + 1, STRIDE):
            if exclude_box is not None:
                er0, er1, ec0, ec1 = exclude_box
                if (r < er1 and r+PATCH_SIZE > er0 and
                    c < ec1 and c+PATCH_SIZE > ec0):
                    continue
            patch_label = label_full[r:r+PATCH_SIZE, c:c+PATCH_SIZE]
            if patch_label.sum() >= MIN_FLOOD:
                flood_pos.append((r, c, True))
            else:
                if not has_too_much_nodata(r, c):
                    nonflood_pos.append((r, c, False))
    n_nf = min(len(nonflood_pos), len(flood_pos) * RATIO)
    idx  = np.random.choice(len(nonflood_pos), n_nf, replace=False)
    return flood_pos, [nonflood_pos[i] for i in idx]

def extract_and_save(splits, patch_dir):
    with rasterio.open(STACK_PATH) as src:
        for split_name, positions in splits.items():
            split_dir = os.path.join(patch_dir, split_name)
            os.makedirs(split_dir, exist_ok=True)
            existing = len([f for f in os.listdir(split_dir) if f.startswith('X_')])
            if existing > 0:
                print(f'  {split_name}: {existing} patches exist — skipping')
                continue
            saved = flood_s = nflood_s = 0
            t0 = time.time()
            for idx, (row, col, is_flood) in enumerate(positions):
                X = src.read(window=Window(col, row, PATCH_SIZE, PATCH_SIZE)).astype(np.float32)
                y = label_full[row:row+PATCH_SIZE, col:col+PATCH_SIZE]
                n_augs = 8 if is_flood else 1
                for k in range(n_augs):
                    Xa, ya = augment(X.copy(), y.copy(), k)
                    np.save(os.path.join(split_dir, f'X_{saved:05d}.npy'), Xa)
                    np.save(os.path.join(split_dir, f'y_{saved:05d}.npy'), ya)
                    saved += 1
                if is_flood: flood_s += 1
                else: nflood_s += 1
                if (idx+1) % 200 == 0:
                    print(f'  {split_name}: {idx+1}/{len(positions)} '
                          f'({saved} patches, {time.time()-t0:.0f}s)')
            gc.collect()
            print(f'  {split_name} DONE: {saved} patches '
                  f'(flood={flood_s}x8={flood_s*8}, nf={nflood_s})')

def zip_splits(patch_dir, zip_dir, tag):
    for split_name in ['train', 'val', 'test']:
        zip_path  = os.path.join(zip_dir, f'patches_{tag}_{split_name}.zip')
        split_dir = os.path.join(patch_dir, split_name)
        if os.path.exists(zip_path):
            print(f'  {split_name}: zip exists — skipping')
            continue
        files = sorted(os.listdir(split_dir))
        print(f'  Zipping {split_name} ({len(files)} files)...')
        t0 = time.time()
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as zf:
            for i, fname in enumerate(files):
                zf.write(os.path.join(split_dir, fname), arcname=fname)
        print(f'  {split_name}: {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s')

def unzip_to_local(zip_dir, tag, local_dir):
    for split in ['train', 'val', 'test']:
        local_split = os.path.join(local_dir, split)
        if os.path.exists(local_split):
            n = len([f for f in os.listdir(local_split) if f.startswith('X_')])
            if n > 0:
                print(f'  {split}: {n:,} patches on disk — skipping')
                continue
        os.makedirs(local_split, exist_ok=True)
        zip_path = os.path.join(zip_dir, f'patches_{tag}_{split}.zip')
        if not os.path.exists(zip_path):
            raise FileNotFoundError(f'{zip_path} not found. Run extraction first.')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(local_split)
        n = len([f for f in os.listdir(local_split) if f.startswith('X_')])
        print(f'  {split}: {n:,} patches')

In [ ]:
class PatchDataset(Dataset):
    def __init__(self, patch_dir, n_channels=12):
        self.patch_dir  = patch_dir
        self.n_channels = n_channels
        self.X_files    = sorted([f for f in os.listdir(patch_dir) if f.startswith('X_')])
    def __len__(self): return len(self.X_files)
    def __getitem__(self, idx):
        num = self.X_files[idx].split('_')[1].split('.')[0]
        X   = np.load(os.path.join(self.patch_dir, f'X_{num}.npy'))
        y   = np.load(os.path.join(self.patch_dir, f'y_{num}.npy'))
        X   = X[:self.n_channels]
        for c in range(X.shape[0]):
            mn, mx = X[c].min(), X[c].max()
            X[c]   = (X[c]-mn)/(mx-mn+1e-8)
        return torch.FloatTensor(X), torch.FloatTensor(y).unsqueeze(0)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels=12):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)
        self.bottleneck = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.up4  = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = DoubleConv(512, 256)
        self.up3  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)
        self.up2  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        self.up1  = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)
        self.out  = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out(d1))

def dice_loss(p, t, eps=1e-6):
    p = p.view(-1); t = t.view(-1)
    return 1-(2*(p*t).sum()+eps)/(p.sum()+t.sum()+eps)

def combined_loss(p, t):
    return F.binary_cross_entropy(p, t) + dice_loss(p, t)

def make_loader(ds, batch_size=16, shuffle=False):
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                       num_workers=0, generator=g)

def train_model(model, train_loader, val_loader, model_dir, name,
                max_epochs=30, patience=7, lr=1e-3):
    opt  = torch.optim.Adam(model.parameters(), lr=lr)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    best = float('inf'); no_imp = 0
    hist = {'train_loss':[], 'val_loss':[]}
    path = os.path.join(model_dir, f'{name}_best.pt')
    for epoch in range(1, max_epochs+1):
        model.train()
        tl = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = combined_loss(model(X), y)
            loss.backward(); opt.step()
            tl += loss.item()
        tl /= len(train_loader)
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                vl += combined_loss(model(X), y).item()
        vl /= len(val_loader)
        hist['train_loss'].append(tl)
        hist['val_loss'].append(vl)
        sch.step(vl)
        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
        if vl < best:
            best = vl; no_imp = 0
            torch.save(model.state_dict(), path)
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break
    model.load_state_dict(torch.load(path))
    print(f'  Best val loss: {best:.4f}')
    return model, hist

def evaluate_model(model, test_dir, n_channels=12):
    ds     = PatchDataset(test_dir, n_channels=n_channels)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    model.eval()
    tp = fp = fn = 0
    all_prob = []; all_true = []
    print(f'  Evaluating {len(ds):,} patches...')
    with torch.no_grad():
        for i, (X, y) in enumerate(loader):
            prob = model(X.to(device)).cpu().numpy().flatten()
            pred = (prob >= 0.5).astype(int)
            true = y.numpy().flatten().astype(int)
            tp  += int(((pred==1)&(true==1)).sum())
            fp  += int(((pred==1)&(true==0)).sum())
            fn  += int(((pred==0)&(true==1)).sum())
            all_prob.extend(prob.tolist())
            all_true.extend(true.tolist())
            del X, prob, pred, true
            if i % 500 == 0:
                torch.cuda.empty_cache(); gc.collect()
    iou = tp/(tp+fp+fn) if (tp+fp+fn)>0 else 0.0
    ap  = np.array(all_prob); at = np.array(all_true)
    return {
        'f1':    f1_score(at, (ap>=0.5).astype(int), zero_division=0),
        'iou':   iou,
        'kappa': cohen_kappa_score(at, (ap>=0.5).astype(int)),
        'auc':   roc_auc_score(at, ap)
    }

def run_direction(tag, train_bounds, test_bounds, exclude_from_test=None, exclude_from_train=None):
    """
    train_bounds / test_bounds: (row_start, row_end, col_start, col_end)
    exclude_from_test / exclude_from_train: optional (r0,r1,c0,c1) box to exclude from that region
    Returns dict with fusion_metrics, optical_metrics, hist for both models.
    """
    patch_dir = os.path.join(PATCH_DIR, tag)
    zip_dir   = ZIP_DIR
    local_dir = os.path.join(LOCAL_DIR, tag)
    model_dir = os.path.join(MODEL_DIR, tag)
    os.makedirs(model_dir, exist_ok=True)

    np.random.seed(SEED)  # reset before position collection so both directions are independently seeded the same way

    print(f'--- {tag}: collecting positions ---')
    tr0, tr1, tc0, tc1 = train_bounds
    train_flood_pos, train_nf_pos = collect_positions(tr0, tr1, tc0, tc1, exclude_box=exclude_from_train)
    all_train = train_flood_pos + train_nf_pos
    print(f'  TRAIN region — flood: {len(train_flood_pos)}  non-flood: {len(train_nf_pos)}')

    te0, te1, tec0, tec1 = test_bounds
    test_flood_pos, test_nf_pos = collect_positions(te0, te1, tec0, tec1, exclude_box=exclude_from_test)
    test_positions = test_flood_pos + test_nf_pos
    print(f'  TEST region  — flood: {len(test_flood_pos)}  non-flood: {len(test_nf_pos)}')

    np.random.shuffle(all_train)
    n_val = int(len(all_train) * 0.15)
    val_positions   = all_train[:n_val]
    train_positions = all_train[n_val:]

    splits = {'train': train_positions, 'val': val_positions, 'test': test_positions}
    print('Final split (before augmentation):')
    for sn, pos in splits.items():
        fn = sum(1 for p in pos if p[2])
        print(f'  {sn:<6}: {len(pos):,} positions ({fn} flood)')

    print(f'--- {tag}: extracting patches ---')
    extract_and_save(splits, patch_dir)

    print(f'--- {tag}: zipping ---')
    zip_splits(patch_dir, zip_dir, tag)

    print(f'--- {tag}: unzipping to local disk ---')
    unzip_to_local(zip_dir, tag, local_dir)

    print(f'--- {tag}: training fusion U-Net (12-channel) ---')
    train_ds = PatchDataset(local_dir+'/train', n_channels=12)
    val_ds   = PatchDataset(local_dir+'/val',   n_channels=12)
    train_loader = make_loader(train_ds, batch_size=16, shuffle=True)
    val_loader   = make_loader(val_ds,   batch_size=16, shuffle=False)
    torch.manual_seed(SEED)
    fusion_model = UNet(in_channels=12).to(device)
    fusion_model, fusion_hist = train_model(fusion_model, train_loader, val_loader,
                                             model_dir, name=f'fusion_unet_{tag}')
    fusion_metrics = evaluate_model(fusion_model, local_dir+'/test', n_channels=12)
    print(f'  Fusion  F1={fusion_metrics["f1"]:.4f} IoU={fusion_metrics["iou"]:.4f} '
          f'Kappa={fusion_metrics["kappa"]:.4f} AUC={fusion_metrics["auc"]:.4f}')

    print(f'--- {tag}: training optical-only U-Net (9-channel) ---')
    train_ds_o = PatchDataset(local_dir+'/train', n_channels=9)
    val_ds_o   = PatchDataset(local_dir+'/val',   n_channels=9)
    train_loader_o = make_loader(train_ds_o, batch_size=16, shuffle=True)
    val_loader_o   = make_loader(val_ds_o,   batch_size=16, shuffle=False)
    torch.manual_seed(SEED)
    optical_model = UNet(in_channels=9).to(device)
    optical_model, optical_hist = train_model(optical_model, train_loader_o, val_loader_o,
                                               model_dir, name=f'optical_unet_{tag}')
    optical_metrics = evaluate_model(optical_model, local_dir+'/test', n_channels=9)
    print(f'  Optical F1={optical_metrics["f1"]:.4f} IoU={optical_metrics["iou"]:.4f} '
          f'Kappa={optical_metrics["kappa"]:.4f} AUC={optical_metrics["auc"]:.4f}')

    return {'fusion': fusion_metrics, 'optical': optical_metrics,
            'fusion_hist': fusion_hist, 'optical_hist': optical_hist}

In [ ]:
class PatchDataset(Dataset):
    def __init__(self, patch_dir, n_channels=12):
        self.patch_dir  = patch_dir
        self.n_channels = n_channels
        self.X_files    = sorted([f for f in os.listdir(patch_dir) if f.startswith('X_')])
    def __len__(self): return len(self.X_files)
    def __getitem__(self, idx):
        num = self.X_files[idx].split('_')[1].split('.')[0]
        X   = np.load(os.path.join(self.patch_dir, f'X_{num}.npy'))
        y   = np.load(os.path.join(self.patch_dir, f'y_{num}.npy'))
        X   = X[:self.n_channels]
        for c in range(X.shape[0]):
            mn, mx = X[c].min(), X[c].max()
            X[c]   = (X[c]-mn)/(mx-mn+1e-8)
        return torch.FloatTensor(X), torch.FloatTensor(y).unsqueeze(0)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels=12):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)
        self.bottleneck = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.up4  = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = DoubleConv(512, 256)
        self.up3  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)
        self.up2  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        self.up1  = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)
        self.out  = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out(d1))

def dice_loss(p, t, eps=1e-6):
    p = p.view(-1); t = t.view(-1)
    return 1-(2*(p*t).sum()+eps)/(p.sum()+t.sum()+eps)

def combined_loss(p, t):
    return F.binary_cross_entropy(p, t) + dice_loss(p, t)

def make_loader(ds, batch_size=16, shuffle=False):
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                       num_workers=0, generator=g)

def train_model(model, train_loader, val_loader, model_dir, name,
                max_epochs=30, patience=7, lr=1e-3):
    path = os.path.join(model_dir, f'{name}_best.pt')
    if os.path.exists(path):
        print(f'  {name}: checkpoint already exists — loading instead of retraining')
        model.load_state_dict(torch.load(path, map_location=device))
        return model, {'train_loss': [], 'val_loss': []}

    opt  = torch.optim.Adam(model.parameters(), lr=lr)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    best = float('inf'); no_imp = 0
    hist = {'train_loss':[], 'val_loss':[]}
    for epoch in range(1, max_epochs+1):
        model.train()
        tl = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = combined_loss(model(X), y)
            loss.backward(); opt.step()
            tl += loss.item()
        tl /= len(train_loader)
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                vl += combined_loss(model(X), y).item()
        vl /= len(val_loader)
        hist['train_loss'].append(tl)
        hist['val_loss'].append(vl)
        sch.step(vl)
        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
        if vl < best:
            best = vl; no_imp = 0
            torch.save(model.state_dict(), path)
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break
    model.load_state_dict(torch.load(path))
    print(f'  Best val loss: {best:.4f}')
    return model, hist

def evaluate_model(model, test_dir, n_channels=12):
    ds = PatchDataset(test_dir, n_channels=n_channels)
    n_patches = len(ds)
    total_pixels = n_patches * PATCH_SIZE * PATCH_SIZE
    all_prob = np.empty(total_pixels, dtype=np.float32)
    all_true = np.empty(total_pixels, dtype=np.uint8)
    ptr = 0

    loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=0)
    model.eval()
    tp = fp = fn = 0
    print(f'  Evaluating {n_patches:,} patches...')
    with torch.no_grad():
        for i, (X, y) in enumerate(loader):
            prob = model(X.to(device)).cpu().numpy().flatten()
            true = y.numpy().flatten().astype(np.uint8)
            pred = (prob >= 0.5).astype(np.uint8)
            tp += int(((pred==1)&(true==1)).sum())
            fp += int(((pred==1)&(true==0)).sum())
            fn += int(((pred==0)&(true==1)).sum())
            n_pix = prob.shape[0]
            all_prob[ptr:ptr+n_pix] = prob
            all_true[ptr:ptr+n_pix] = true
            ptr += n_pix
            del X, prob, pred, true
            if i % 200 == 0:
                torch.cuda.empty_cache(); gc.collect()

    all_prob = all_prob[:ptr]
    all_true = all_true[:ptr]
    iou = tp/(tp+fp+fn) if (tp+fp+fn)>0 else 0.0
    pred_bin = (all_prob >= 0.5).astype(np.uint8)
    metrics = {
        'f1':    f1_score(all_true, pred_bin, zero_division=0),
        'iou':   iou,
        'kappa': cohen_kappa_score(all_true, pred_bin),
        'auc':   roc_auc_score(all_true, all_prob)
    }
    del all_prob, all_true, pred_bin
    gc.collect()
    return metrics

def run_direction(tag, train_bounds, test_bounds, exclude_from_test=None, exclude_from_train=None):
    """
    train_bounds / test_bounds: (row_start, row_end, col_start, col_end)
    exclude_from_test / exclude_from_train: optional (r0,r1,c0,c1) box to exclude from that region
    Returns dict with fusion_metrics, optical_metrics, hist for both models.
    """
    patch_dir = os.path.join(PATCH_DIR, tag)
    zip_dir   = ZIP_DIR
    local_dir = os.path.join(LOCAL_DIR, tag)
    model_dir = os.path.join(MODEL_DIR, tag)
    os.makedirs(model_dir, exist_ok=True)

    np.random.seed(SEED)

    print(f'--- {tag}: collecting positions ---')
    tr0, tr1, tc0, tc1 = train_bounds
    train_flood_pos, train_nf_pos = collect_positions(tr0, tr1, tc0, tc1, exclude_box=exclude_from_train)
    all_train = train_flood_pos + train_nf_pos
    print(f'  TRAIN region — flood: {len(train_flood_pos)}  non-flood: {len(train_nf_pos)}')

    te0, te1, tec0, tec1 = test_bounds
    test_flood_pos, test_nf_pos = collect_positions(te0, te1, tec0, tec1, exclude_box=exclude_from_test)
    test_positions = test_flood_pos + test_nf_pos
    print(f'  TEST region  — flood: {len(test_flood_pos)}  non-flood: {len(test_nf_pos)}')

    np.random.shuffle(all_train)
    n_val = int(len(all_train) * 0.15)
    val_positions   = all_train[:n_val]
    train_positions = all_train[n_val:]

    splits = {'train': train_positions, 'val': val_positions, 'test': test_positions}
    print('Final split (before augmentation):')
    for sn, pos in splits.items():
        fn = sum(1 for p in pos if p[2])
        print(f'  {sn:<6}: {len(pos):,} positions ({fn} flood)')

    print(f'--- {tag}: extracting patches ---')
    extract_and_save(splits, patch_dir)

    print(f'--- {tag}: zipping ---')
    zip_splits(patch_dir, zip_dir, tag)

    print(f'--- {tag}: unzipping to local disk ---')
    unzip_to_local(zip_dir, tag, local_dir)

    print(f'--- {tag}: training fusion U-Net (12-channel) ---')
    train_ds = PatchDataset(local_dir+'/train', n_channels=12)
    val_ds   = PatchDataset(local_dir+'/val',   n_channels=12)
    train_loader = make_loader(train_ds, batch_size=16, shuffle=True)
    val_loader   = make_loader(val_ds,   batch_size=16, shuffle=False)
    torch.manual_seed(SEED)
    fusion_model = UNet(in_channels=12).to(device)
    fusion_model, fusion_hist = train_model(fusion_model, train_loader, val_loader,
                                             model_dir, name=f'fusion_unet_{tag}')
    fusion_metrics = evaluate_model(fusion_model, local_dir+'/test', n_channels=12)
    print(f'  Fusion  F1={fusion_metrics["f1"]:.4f} IoU={fusion_metrics["iou"]:.4f} '
          f'Kappa={fusion_metrics["kappa"]:.4f} AUC={fusion_metrics["auc"]:.4f}')

    print(f'--- {tag}: training optical-only U-Net (9-channel) ---')
    train_ds_o = PatchDataset(local_dir+'/train', n_channels=9)
    val_ds_o   = PatchDataset(local_dir+'/val',   n_channels=9)
    train_loader_o = make_loader(train_ds_o, batch_size=16, shuffle=True)
    val_loader_o   = make_loader(val_ds_o,   batch_size=16, shuffle=False)
    torch.manual_seed(SEED)
    optical_model = UNet(in_channels=9).to(device)
    optical_model, optical_hist = train_model(optical_model, train_loader_o, val_loader_o,
                                               model_dir, name=f'optical_unet_{tag}')
    optical_metrics = evaluate_model(optical_model, local_dir+'/test', n_channels=9)
    print(f'  Optical F1={optical_metrics["f1"]:.4f} IoU={optical_metrics["iou"]:.4f} '
          f'Kappa={optical_metrics["kappa"]:.4f} AUC={optical_metrics["auc"]:.4f}')

    return {'fusion': fusion_metrics, 'optical': optical_metrics,
            'fusion_hist': fusion_hist, 'optical_hist': optical_hist}

---
---
# Direction B — Northern Channels Train / Harbour Basin Test

- **TRAIN** : northern channels box (12.9% of flood pixels)
- **TEST**  : harbour basin — full flood region minus the northern channels box (87.1%)


In [ ]:
COL_BUFFER = 256
col_start_full = max(0, FLOOD_COL_MIN - COL_BUFFER)
col_end_full   = min(W, FLOOD_COL_MAX + COL_BUFFER)

results_B = run_direction(
    tag='dirB_north_train_harbour_test',
    train_bounds=(NORTH_R0, NORTH_R1, NORTH_C0, NORTH_C1),
    test_bounds=(FLOOD_ROW_MIN, FLOOD_ROW_MAX, col_start_full, col_end_full),
    exclude_from_train=None,
    exclude_from_test=(NORTH_R0, NORTH_R1, NORTH_C0, NORTH_C1)
)

--- dirB_north_train_harbour_test: collecting positions ---
  TRAIN region — flood: 411  non-flood: 343
  TEST region  — flood: 452  non-flood: 4520
Final split (before augmentation):
  train : 641 positions (354 flood)
  val   : 113 positions (57 flood)
  test  : 4,972 positions (452 flood)
--- dirB_north_train_harbour_test: extracting patches ---
  train: 3119 patches exist — skipping
  val: 512 patches exist — skipping
  test: 8136 patches exist — skipping
--- dirB_north_train_harbour_test: zipping ---
  train: zip exists — skipping
  val: zip exists — skipping
  test: zip exists — skipping
--- dirB_north_train_harbour_test: unzipping to local disk ---
  train: 3,119 patches on disk — skipping
  val: 512 patches on disk — skipping
  test: 8,136 patches on disk — skipping
--- dirB_north_train_harbour_test: training fusion U-Net (12-channel) ---
  fusion_unet_dirB_north_train_harbour_test: checkpoint already exists — loading instead of retraining
  Evaluating 8,136 patches...
  Fusion

---
---
# Direction A — Harbour Basin Train / Northern Channels Test

- **TRAIN** : harbour basin — full flood region minus the northern channels box (87.1%)
- **TEST**  : northern channels box (12.9% of flood pixels)




In [ ]:
results_A = run_direction(
    tag='dirA_harbour_train_north_test',
    train_bounds=(FLOOD_ROW_MIN, FLOOD_ROW_MAX, col_start_full, col_end_full),
    test_bounds=(NORTH_R0, NORTH_R1, NORTH_C0, NORTH_C1),
    exclude_from_train=(NORTH_R0, NORTH_R1, NORTH_C0, NORTH_C1),
    exclude_from_test=None
)

--- dirA_harbour_train_north_test: collecting positions ---
  TRAIN region — flood: 452  non-flood: 4520
  TEST region  — flood: 411  non-flood: 343
Final split (before augmentation):
  train : 4,227 positions (385 flood)
  val   : 745 positions (67 flood)
  test  : 754 positions (411 flood)
--- dirA_harbour_train_north_test: extracting patches ---
  train: 200/4227 (305 patches, 54s)
  train: 400/4227 (638 patches, 77s)
  train: 600/4227 (978 patches, 100s)
  train: 800/4227 (1269 patches, 122s)
  train: 1000/4227 (1609 patches, 144s)
  train: 1200/4227 (1928 patches, 170s)
  train: 1400/4227 (2261 patches, 193s)
  train: 1600/4227 (2587 patches, 214s)
  train: 1800/4227 (2906 patches, 236s)
  train: 2000/4227 (3218 patches, 257s)
  train: 2200/4227 (3523 patches, 279s)
  train: 2400/4227 (3842 patches, 302s)
  train: 2600/4227 (4161 patches, 324s)
  train: 2800/4227 (4501 patches, 345s)
  train: 3000/4227 (4883 patches, 370s)
  train: 3200/4227 (5258 patches, 394s)
  train: 3400/4227

---
## Final Summary — Table 4.3, Both Rows

In [ ]:
print('=' * 78)
print('SPATIAL BLOCK TRANSFERABILITY — BOTH DIRECTIONS (Table 4.3)')
print('=' * 78)
print()
print(f'{"":<45} {"F1":>7} {"IoU":>7} {"Kappa":>7} {"AUC":>7}')

print()
print('Direction B — Northern channels train / Harbour basin test (fusion, 12-ch):')
m = results_B["fusion"]
print(f'  {"Fusion":<43} {m["f1"]:>7.4f} {m["iou"]:>7.4f} {m["kappa"]:>7.4f} {m["auc"]:>7.4f}')
m = results_B["optical"]
print(f'  {"Optical":<43} {m["f1"]:>7.4f} {m["iou"]:>7.4f} {m["kappa"]:>7.4f} {m["auc"]:>7.4f}')

print()
print('Direction A — Harbour basin train / Northern channels test (fusion, 12-ch):')
m = results_A["fusion"]
print(f'  {"Fusion":<43} {m["f1"]:>7.4f} {m["iou"]:>7.4f} {m["kappa"]:>7.4f} {m["auc"]:>7.4f}')
m = results_A["optical"]
print(f'  {"Optical":<43} {m["f1"]:>7.4f} {m["iou"]:>7.4f} {m["kappa"]:>7.4f} {m["auc"]:>7.4f}')



print('Models saved under:', MODEL_DIR)
print('Figures saved under:', FIG_DIR)


SPATIAL BLOCK TRANSFERABILITY — BOTH DIRECTIONS (Table 4.3)

                                                   F1     IoU   Kappa     AUC

Direction B — Northern channels train / Harbour basin test (fusion, 12-ch):
  Fusion                                       0.2495  0.1425  0.2269  0.6947
  Optical                                      0.1208  0.0643  0.1048  0.4724

Direction A — Harbour basin train / Northern channels test (fusion, 12-ch):
  Fusion                                       0.3868  0.2398  0.3764  0.8284
  Optical                                      0.3812  0.2355  0.3735  0.8300
Models saved under: /content/drive/MyDrive/KZN_Research_Colab/Models_combined_split/
Figures saved under: /content/drive/MyDrive/KZN_Research_Colab/Chapter4_Figures_combined_split/
